In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process.kernels import (
    WhiteKernel, ConstantKernel, RBF, RationalQuadratic, ExpSineSquared, DotProduct
)
from auto_kernel_ensemble import KernelAutoregressiveModel
from data_loader import DataLoader
from walk_forward import WalkForwardValidator
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from metrics import ForecastingMetrics

## Mean Reversion Analysis
- Analyze crossover of historical mean and median. What is the optimimal window size to capture reversion. Is it enough to be profitable? How often does the time series ocilate around the mean?
- How do you statistically model reversions to the mean? Expected peak or valley before reversion?

In [4]:
# load the data
# read in the data for the below ticker
ticker = 'OSK'
df_asset = pd.read_parquet(f'data/{ticker}.parquet')
df_asset.head()

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
1985-10-02,1.359799,2.791667,2.833333,2.791667,2.791667,4849200
1985-10-03,1.380093,2.833333,2.875000,2.833333,2.833333,1974000
1985-10-04,1.380093,2.833333,2.875000,2.833333,2.833333,694800
1985-10-07,1.359799,2.791667,2.833333,2.791667,2.833333,337200
1985-10-08,1.359799,2.791667,2.833333,2.791667,2.791667,436200


In [5]:
# split the dat into train and test sets
split_record = int(len(df_asset)*0.8)
df_asset_train = df_asset.iloc[:split_record]
df_asset_test = df_asset.iloc[split_record:]

# split the training into analysis and validation sets
split_record2 = int(len(df_asset_train)*0.8)
df_asset_train_analysis = df_asset_train.iloc[:split_record2]
df_asset_train_validation = df_asset_train.iloc[split_record2:]
print (
    f'Train set size: {len(df_asset_train)}\n'
    f'Test set size: {len(df_asset_test)}\n'
    f'Train set analysis size: {len(df_asset_train_analysis)}\n'
    f'Train set validation size: {len(df_asset_train_validation)}'
)

Train set size: 8099
Test set size: 2025
Train set analysis size: 6479
Train set validation size: 1620


## Ociilating Around the Mean

Capture the following statistics around the raw adjusted close price and a historic running mean or median value.

- Number of times the price crosses the historic mean
- Number of days between each crossover (statistics and distribution)
- magnitude, the higest peak and lowest trough before reversion back to the mean


In [ ]:
# create a class function that take